# COMP5318 Assignment 1: Rice Classification

##### Group number: 64
##### Student 1 SID: 540935875
##### Student 2 SID: 550087894
##### Student 3 SID: 520271713
##### Student 4 SID: ... 

## **1. Data Pre-processing**

In [14]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

In [15]:
# Load the rice dataset. Change only this filename to test another compatible CSV.
DATASET_PATH = Path("rice-final2.csv")
dataset = pd.read_csv(DATASET_PATH, na_values=["?"])
dataset.head()

,Area,Perimiter,Major_Axis_Length,Minor_Axis_Length,Eccentricity,Convex_Area,Extent,class
0,12573.0,461.466003,192.903351,84.572075,0.898772,12893.0,0.550433,class2
1,12845.0,464.121002,194.332214,85.524338,0.897952,13125.0,0.774962,class2
2,14055.0,488.748993,207.751755,87.250328,0.907536,14484.0,0.550076,class1
3,14412.0,490.324005,207.476135,89.689514,0.901735,14703.0,0.598853,class1
4,14658.0,477.117004,189.566635,99.997780,0.849551,15048.0,0.649504,class2


In [16]:
def preprocess_dataset(data):
    """Use every column except the last as features; the last column is the label."""
    features = data.iloc[:, :-1].apply(pd.to_numeric, errors="raise")
    labels = data.iloc[:, -1].astype(str).str.strip()
    encoded_labels = labels.map({"class1": 0, "class2": 1})
    if encoded_labels.isna().any():
        raise ValueError("The last column must contain only class1 and class2.")

    # Keeping empty features preserves dimensionality for an all-missing column.
    # Such a column has no observed mean; SimpleImputer retains it as zeros.
    imputer = SimpleImputer(strategy="mean", keep_empty_features=True)
    imputed_features = imputer.fit_transform(features)

    scaler = MinMaxScaler(feature_range=(0, 1))
    X_processed = scaler.fit_transform(imputed_features)
    y_processed = encoded_labels.to_numpy(dtype=int)
    return X_processed, y_processed


X, y = preprocess_dataset(dataset)

In [17]:
# Print only the first ten rows, with four decimals for features and integer labels.
def print_data(X, y, n_rows=10):
    """Takes a numpy data array and target and prints the first ten rows.
    
    Arguments:
        X: numpy array of shape (n_examples, n_features)
        y: numpy array of shape (n_examples)
        n_rows: numpy of rows to print
    """
    for example_num in range(n_rows):
        for feature in X[example_num]:
            print("{:.4f}".format(feature), end=",")

        if example_num == len(X)-1:
            print(y[example_num],end="")
        else:
            print(y[example_num])


print_data(X, y)

0.4628,0.5406,0.5113,0.4803,0.7380,0.4699,0.1196,1
0.4900,0.5547,0.5266,0.5018,0.7319,0.4926,0.8030,1
0.6109,0.6847,0.6707,0.5409,0.8032,0.6253,0.1185,0
0.6466,0.6930,0.6677,0.5961,0.7601,0.6467,0.2669,0
0.6712,0.6233,0.4755,0.8293,0.3721,0.6803,0.4211,1
0.2634,0.2932,0.2414,0.4127,0.5521,0.2752,0.2825,1
0.8175,0.9501,0.9515,0.5925,0.9245,0.8162,0.0000,0
0.3174,0.3588,0.3601,0.3908,0.6921,0.3261,0.8510,1
0.3130,0.3050,0.2150,0.5189,0.3974,0.3159,0.4570,1
0.5120,0.5237,0.4409,0.6235,0.5460,0.5111,0.3155,1


## **2. Build Classifiers**

- Part 1:  Logistic Regression, Naïve Bayes
- Part 2:  KNN, Decision Tree, Ada Boost, Gradient Boost, Random Forest, SVM

### Part 1: Cross-validation without parameter tuning

In [18]:
# Use exactly the stratified ten-fold configuration specified in the assignment.
cvKFold = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

In [19]:
# Logistic Regression: no parameter search in Part 1.
# A higher iteration limit allows the default solver to converge reliably.
logistic_model = LogisticRegression(random_state=0, max_iter=1000)
logistic_scores = cross_val_score(
    logistic_model, X, y, cv=cvKFold, scoring="accuracy", error_score="raise",
)
logistic_cv_accuracy = logistic_scores.mean()

In [20]:
# Gaussian Naive Bayes is suitable for continuous numerical features.
# GaussianNB has no random_state parameter because its fitting is deterministic.
nb_model = GaussianNB()
nb_scores = cross_val_score(
    nb_model, X, y, cv=cvKFold, scoring="accuracy", error_score="raise",
)
nb_cv_accuracy = nb_scores.mean()

### Part 1 Results


In [21]:
part1_results = {
    "LogR": {"cv_accuracy": float(logistic_cv_accuracy)},
    "NB": {"cv_accuracy": float(nb_cv_accuracy)},
}
print(f"LogR average cross-validation accuracy: {logistic_cv_accuracy:.4f}")
print(f"NB average cross-validation accuracy: {nb_cv_accuracy:.4f}")

LogR average cross-validation accuracy: 0.9386
NB average cross-validation accuracy: 0.9264


### Part 2: Cross-validation with parameter tuning

In [22]:
# KNN: k neighbours; p=1 means Manhattan distance, p=2 Euclidean distance.
k = [1, 3, 5, 7]
p = [1, 2]
knn_model = KNeighborsClassifier()
knn_param_grid = {"n_neighbors": k, "p": p}

In [23]:
# Decision Tree
max_depth = [3, 5, 7, 10]
min_samples_split = [2, 5, 10]
min_samples_leaf = [1, 2, 4]
dt_model = DecisionTreeClassifier(random_state=0)
dt_param_grid = {
    "max_depth": max_depth,
    "min_samples_split": min_samples_split,
    "min_samples_leaf": min_samples_leaf,
}

In [24]:
# Ada Boost
n_estimators = [50, 100, 150]
learning_rate = [0.1, 0.2, 0.3, 0.5]
ada_model = AdaBoostClassifier(random_state=0)
ada_param_grid = {"n_estimators": n_estimators, "learning_rate": learning_rate}

In [25]:
# Gradient Boosting
max_depth = [1, 3, 5, 7]
n_estimators = [50, 100, 150]
learning_rate = [0.1, 0.2, 0.3, 0.5]
gb_model = GradientBoostingClassifier(random_state=0)
gb_param_grid = {
    "max_depth": max_depth,
    "n_estimators": n_estimators,
    "learning_rate": learning_rate,
}

In [26]:
# Random Forest: entropy implements information gain; sqrt is required.
n_estimators = [10, 30, 60, 100]
max_leaf_nodes = [6, 12]
rf_model = RandomForestClassifier(
    criterion="entropy", max_features="sqrt", random_state=0,
)
rf_param_grid = {"n_estimators": n_estimators, "max_leaf_nodes": max_leaf_nodes}

In [27]:
# SVM: use RBF because both C and gamma affect this kernel.
C = [0.01, 0.1, 1, 5]
gamma = [0.01, 0.1, 1, 10]
svm_model = SVC(kernel="rbf", random_state=0)
svm_param_grid = {"C": C, "gamma": gamma}

### Part 2: Results

In [29]:
# Split once, so every Part 2 classifier is evaluated on the same examples.
# Use the default split ration 0.75 : 0.25.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0,
)

def tune_and_evaluate(model, param_grid, X_train, X_test, y_train, y_test):
    """Tune on the training set, then score the refitted winner on the test set."""
    baseline_scores = cross_val_score(
        model, X_train, y_train, cv=cvKFold, scoring="accuracy", error_score="raise",
    )
    search = GridSearchCV(
        estimator=model, param_grid=param_grid, scoring="accuracy", cv=cvKFold, refit=True, error_score="raise",
    )
    search.fit(X_train, y_train)
    predictions = search.predict(X_test)
    result = {
        "best_params": search.best_params_,
        "baseline_cv_accuracy": float(baseline_scores.mean()),
        "cv_accuracy": float(search.best_score_),
        "test_accuracy": float(accuracy_score(y_test, predictions)),
    }
    return search, predictions, result


model_specs = {
    "KNN": (knn_model, knn_param_grid),
    "DT": (dt_model, dt_param_grid),
    "AdaBoost": (ada_model, ada_param_grid),
    "GB": (gb_model, gb_param_grid),
    "RF": (rf_model, rf_param_grid),
    "SVM": (svm_model, svm_param_grid),
}
searches, test_predictions, part2_results = {}, {}, {}

for name, (model, param_grid) in model_specs.items():
    search, predictions, result = tune_and_evaluate(
        model, param_grid, X_train, X_test, y_train, y_test,
    )
    searches[name] = search
    test_predictions[name] = predictions
    part2_results[name] = result

    for parameter, value in result["best_params"].items():
        label = "k" if parameter == "n_neighbors" else parameter
        # C, gamma and learning_rate are real-valued even when a grid entry is 1.
        formatted = f"{value:.4f}" if parameter in {"C", "gamma", "learning_rate"} else str(value)
        print(f"{name} best {label}: {formatted}")
    print(f"{name} cross-validation accuracy: {result['cv_accuracy']:.4f}")
    print(f"{name} test set accuracy: {result['test_accuracy']:.4f}")

    if name == "RF":
        result["macro_f1"] = float(f1_score(y_test, predictions, average="macro", zero_division=0))
        result["weighted_f1"] = float(f1_score(y_test, predictions, average="weighted", zero_division=0))
        print(f"RF test set macro average F1: {result['macro_f1']:.4f}")
        print(f"RF test set weighted average F1: {result['weighted_f1']:.4f}")
    print()

KNN best k: 5
KNN best p: 1
KNN cross-validation accuracy: 0.9371
KNN test set accuracy: 0.9257

DT best max_depth: 5
DT best min_samples_leaf: 1
DT best min_samples_split: 5
DT cross-validation accuracy: 0.9314
DT test set accuracy: 0.9400

AdaBoost best learning_rate: 0.5000
AdaBoost best n_estimators: 150
AdaBoost cross-validation accuracy: 0.9438
AdaBoost test set accuracy: 0.9429

GB best learning_rate: 0.3000
GB best max_depth: 1
GB best n_estimators: 50
GB cross-validation accuracy: 0.9448
GB test set accuracy: 0.9457

RF best max_leaf_nodes: 12
RF best n_estimators: 30
RF cross-validation accuracy: 0.9390
RF test set accuracy: 0.9371
RF test set macro average F1: 0.9355
RF test set weighted average F1: 0.9370

SVM best C: 5.0000
SVM best gamma: 1.0000
SVM cross-validation accuracy: 0.9457
SVM test set accuracy: 0.9343



### Test your code

To check another compatible dataset, change `DATASET_PATH` in the loading cell
and rerun from the beginning. Restore `rice-final2.csv` and rerun before submission.
The supplied `test-before.csv` was also executed separately; no results from that
dataset are saved in this notebook.

## **3. Reflection and Discussion**



### Performance comparison

| Classifier | Mean / best CV accuracy | Test accuracy |
| --- | ---: | ---: |
| Logistic Regression (Part 1) | 0.9386 | Not evaluated |
| Gaussian Naïve Bayes (Part 1) | 0.9264 | Not evaluated |
| KNN (Part 2) | 0.9371 | 0.9257 |
| Decision Tree (Part 2) | 0.9314 | 0.9400 |
| AdaBoost (Part 2) | 0.9438 | 0.9429 |
| Gradient Boosting (Part 2) | 0.9448 | 0.9457 |
| Random Forest (Part 2) | 0.9390 | 0.9371 |
| RBF SVM (Part 2) | 0.9457 | 0.9343 |

Logistic Regression outperformed GaussianNB in Part 1, providing a strong simple
baseline. Dependencies among grain measurements and imperfect Gaussian
assumptions may limit Naïve Bayes, although these explanations were not isolated
experimentally. Part 1 evaluates all 1400 examples through CV, whereas Part 2 CV
uses only the 1050-example training subset, so their CV scores are not a strictly·
matched comparison.

Among the tuned models, SVM achieved the highest training-set CV score, while
Gradient Boosting performed best on the 350-example test subset. Gradient
Boosting, AdaBoost and Decision Tree correctly classified 331, 330 and 329 test
examples, respectively. These small gaps do not establish a statistically
significant ranking. Random Forest obtained macro F1 of 0.9355 and weighted F1 of
0.9370; its class-specific F1 scores were 0.9252 for class1 and 0.9458 for class2.
The larger, better-predicted class2 receives more weight in weighted F1.

### Effect of tuning

Using the same Part 2 training subset and folds, baseline-to-tuned CV accuracy
changed from 0.9333 to 0.9371 for KNN, 0.9143 to 0.9314 for Decision Tree,
0.9400 to 0.9438 for AdaBoost, 0.9362 to 0.9448 for Gradient Boosting,
0.9381 to 0.9390 for Random Forest, and 0.9438 to 0.9457 for SVM. Decision Tree
benefited most. Its selected depth of 5, and Gradient Boosting's depth-1 trees,
are consistent with useful complexity control. Random Forest's improvement was
small. Some DT and RF candidates tied at the maximum, so the returned parameters
are not unique winners. The baseline preserves the required RF entropy/sqrt
settings and SVM's RBF kernel. Tuning gains are specific to these grids and folds;
they do not guarantee improved performance on future data.

### Limitations and improvements

The test proportion was set to the default 0.25 because the supplied specification
does not state one. Parameters were chosen using training CV, not test accuracy.
However, the assignment's preprocessing-first order fits means and scaling
extrema on the whole dataset, allowing validation/test feature information into
preprocessing. Therefore these are assignment-protocol scores, not fully
leakage-free estimates. A separate rigorous evaluation should place imputation
and scaling inside a Pipeline fitted within each CV training fold. Repeated
splits or nested CV would also help assess variability and selection bias. This
experiment does not quantify the leakage effect or establish a universally best
classifier.


## **AI Acknowledgement**